# Fairness Audit and Debiasing for Fraud Detection
This notebook demonstrates pre-processing, model-level debiasing, post-processing calibration, and a fairness compliance report for the fraud detection engine.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from aif360.algorithms.preprocessing import Reweighing
from aif360.datasets import BinaryLabelDataset
from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate, true_positive_rate
from fairlearn.postprocessing import ThresholdOptimizer
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

from data_pipeline import prepare_training_sample
from feature_store import extract_feature_columns
from bias_mitigation import add_location_group, location_group_from_country


In [ ]:
# Protected demographic groups
protected_countries = {
    "high_risk": ["IN", "CN", "RU", "BR", "NG", "PK"],
    "low_risk": ["US", "GB", "DE", "FR", "CA", "AU"],
}

protected_map = {}
for group, countries in protected_countries.items():
    for country in countries:
        protected_map[country] = group

print("Protected group mapping sample:", {k: protected_map[k] for k in list(protected_map)[:4]})


In [ ]:
# Load synthetic training dataset and create protected group labels
df = prepare_training_sample(1000)
df = add_location_group(df)
df["protected_group"] = df["location_country"].apply(location_group_from_country)

print(df["protected_group"].value_counts())
print(df.head(2).T)


In [ ]:
# AIF360 Reweighing pre-processing
labels = df["is_fraud"].astype(int)
protected_feature = df["protected_group"]

binary_dataset = BinaryLabelDataset(
    df=df,
    label_names=["is_fraud"],
    protected_attribute_names=["protected_group"],
    favorable_label=0,
    unfavorable_label=1,
)

reweigher = Reweighing(protected_attribute_names=["protected_group"], favorable_label=0, unfavorable_label=1)
rew_dataset = reweigher.fit_transform(binary_dataset)
weights = rew_dataset.instance_weights
print("Weight stats", weights.min(), weights.max(), weights.mean())


In [ ]:
# Train baseline XGBoost and CatBoost with adversarial debiasing sample weights
feature_cols = extract_feature_columns()
X = df[feature_cols].astype(float)
y = df["is_fraud"].astype(int)
protected = df["protected_group"].astype(str)

# Use AIF360 weights for fairness-aware training
sample_weights = weights

xgb = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
catboost = CatBoostClassifier(iterations=150, learning_rate=0.06, depth=5, verbose=False, random_seed=42)

xgb.fit(X, y, sample_weight=sample_weights)
catboost.fit(X, y, sample_weight=sample_weights)

print("Baseline models trained with reweighing.")


In [ ]:
# Evaluate baseline fairness and performance
xgb_preds = xgb.predict(X)
cat_preds = catboost.predict(X)

mf_xgb_tpr = MetricFrame(true_positive_rate, y, xgb_preds, sensitive_features=protected).by_group
mf_xgb_fpr = MetricFrame(false_positive_rate, y, xgb_preds, sensitive_features=protected).by_group
mf_cat_tpr = MetricFrame(true_positive_rate, y, cat_preds, sensitive_features=protected).by_group
mf_cat_fpr = MetricFrame(false_positive_rate, y, cat_preds, sensitive_features=protected).by_group

print("XGBoost TPR by group:\n", mf_xgb_tpr)
print("XGBoost FPR by group:\n", mf_xgb_fpr)
print("CatBoost TPR by group:\n", mf_cat_tpr)
print("CatBoost FPR by group:\n", mf_cat_fpr)
print("XGBoost F1", f1_score(y, xgb_preds))
print("CatBoost F1", f1_score(y, cat_preds))


In [ ]:
# Apply Fairlearn Equalized Odds post-processing to CatBoost

threshold_optimizer = ThresholdOptimizer(
    estimator=catboost,
    constraints="equalized_odds",
    predict_method="predict_proba",
    prefit=True,
)
threshold_optimizer.fit(X, y, sensitive_features=protected)
post_preds = threshold_optimizer.predict(X, sensitive_features=protected)

print("Calibrated post-processed F1:", f1_score(y, post_preds))
print("Post-processed TPR by group:\n", MetricFrame(true_positive_rate, y, post_preds, sensitive_features=protected).by_group)
print("Post-processed FPR by group:\n", MetricFrame(false_positive_rate, y, post_preds, sensitive_features=protected).by_group)


In [ ]:
# Generate fairness audit report with MetricFrame

def theil_index(y_pred, group):
    favorable = 1 - np.array(y_pred, dtype=float)
    overall = favorable.mean()
    groups = np.unique(group)
    theil = 0.0
    for g in groups:
        mask = group == g
        p = favorable[mask].mean()
        if p > 0:
            theil += (mask.sum() / len(group)) * (p / overall) * np.log((p / overall) + 1e-9)
    return theil

post_fav = 1 - post_preds

disp_impact = MetricFrame(selection_rate, np.zeros_like(post_fav), post_fav, sensitive_features=protected).by_group
print("Disparate Impact Ratio:", disp_impact.min() / disp_impact.max())
print("Equalized Odds Difference:", (MetricFrame(true_positive_rate, y, post_preds, sensitive_features=protected).by_group.max() - MetricFrame(true_positive_rate, y, post_preds, sensitive_features=protected).by_group.min()) + (MetricFrame(false_positive_rate, y, post_preds, sensitive_features=protected).by_group.max() - MetricFrame(false_positive_rate, y, post_preds, sensitive_features=protected).by_group.min()))
print("Theil Index:", theil_index(post_preds, protected))


In [ ]:
# Benchmark FastAPI inference latency for fairness-aware scoring
import time

from fastapi.testclient import TestClient
from api.main import app

client = TestClient(app)
payload = {
    "transaction_id": "TX-TEST-001",
    "timestamp": pd.Timestamp.utcnow().isoformat(),
    "amount": 129.99,
    "merchant_category": "electronics",
    "merchant_id": "M-101",
    "customer_id": "C-3001",
    "location": {"country": "US", "latitude": 37.77, "longitude": -122.42},
    "device": {"device_type": "mobile", "device_id": "device_a", "browser": "chrome"},
    "is_fraud": 0,
}

latencies = []
for _ in range(5):
    start = time.perf_counter()
    response = client.post("/score", json=payload)
    end = time.perf_counter()
    latencies.append((end - start) * 1000)
    assert response.status_code == 200

print("Inference latencies (ms):", latencies)
print("Median latency (ms):", np.median(latencies))
